# 5. Generative Models: Variational Autoencoders

Generative models learn the data distribution $p(x)$ to generate new samples. This notebook covers:
- The **ELBO** loss function derivation
- **Variational Autoencoders (VAE)** with the reparameterization trick
- Training a VAE on MNIST and exploring the latent space

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print(f"Using device: {device}")

## 5.1 The VAE Framework

A VAE introduces a latent variable $z$ and optimizes the **Evidence Lower Bound (ELBO)**:

$$\log p(x) \geq \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{regularization}}$$

- $q(z|x) = \mathcal{N}(\mu(x), \sigma^2(x))$: **encoder**
- $p(x|z)$: **decoder**
- $p(z) = \mathcal{N}(0, I)$: **prior**

**Reparameterization trick**: $z = \mu + \sigma \odot \varepsilon$, $\varepsilon \sim \mathcal{N}(0, I)$ makes sampling differentiable.

KL closed form: $D_{KL} = -\frac{1}{2}\sum_j (1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2)$

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128)

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim), nn.Sigmoid()
        )
    
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 784))
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    recon = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kl

vae = VAE(latent_dim=2).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)
print(f"Parameters: {sum(p.numel() for p in vae.parameters()):,}")

In [ ]:
# Training
n_epochs = 15
train_losses = []
for epoch in range(n_epochs):
    vae.train()
    total_loss = 0
    for images, _ in train_loader:
        images = images.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = vae(images)
        loss = vae_loss(recon, images, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_data)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1:>2}/{n_epochs}, Loss: {avg_loss:.2f}")

plt.figure(figsize=(7, 4))
plt.plot(train_losses, 'o-')
plt.xlabel('Epoch')
plt.ylabel('Loss per sample')
plt.title('VAE Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.3 Reconstructions and Latent Space

In [ ]:
vae.eval()
test_images, _ = next(iter(test_loader))
with torch.no_grad():
    recon, _, _ = vae(test_images.to(device))
recon = recon.cpu().view(-1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(test_images[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i], cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('Reconstructed', fontsize=11)
plt.suptitle('VAE Reconstructions')
plt.tight_layout()
plt.show()

In [ ]:
# Latent space manifold: decode a grid of z values
n = 15
grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)
canvas = np.zeros((28 * n, 28 * n))
vae.eval()
with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            decoded = vae.decode(z).cpu().view(28, 28).numpy()
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = decoded

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(canvas, cmap='gray')
ax.set_title('VAE Latent Space (2D manifold)', fontsize=14)
ax.set_xticks(np.arange(n) * 28 + 14)
ax.set_xticklabels([f'{x:.1f}' for x in grid_x], fontsize=7)
ax.set_yticks(np.arange(n) * 28 + 14)
ax.set_yticklabels([f'{y:.1f}' for y in grid_y], fontsize=7)
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$')
plt.tight_layout()
plt.show()

In [ ]:
# Color-coded latent space by digit class
z_all, labels_all = [], []
with torch.no_grad():
    for images, labels in test_loader:
        mu, _ = vae.encode(images.view(-1, 784).to(device))
        z_all.append(mu.cpu().numpy())
        labels_all.append(labels.numpy())
z_all = np.concatenate(z_all)
labels_all = np.concatenate(labels_all)

fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(z_all[:, 0], z_all[:, 1], c=labels_all, cmap='tab10', s=5, alpha=0.6)
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$')
ax.set_title('Latent Space Colored by Digit Class')
plt.colorbar(scatter, label='Digit')
plt.tight_layout()
plt.show()

## Key Takeaways

- A **VAE** learns a smooth, continuous latent space from which we can sample new data
- The **ELBO** balances reconstruction quality and latent space regularity (KL term)
- The **reparameterization trick** makes the sampling operation differentiable
- With `latent_dim=2`, we can directly visualize how the model organizes digit classes
- VAEs produce slightly blurry samples compared to GANs, but offer stable training and a proper likelihood objective